# 🧪 Agent Engine Provider Test

### 🔁 Step 1: Reset and Seed Agent Engine Providers

In [2]:
from pathlib import Path
import os, sys

# Set working directory explicitly to project root
PROJECT_ROOT = Path(r"C:\Repos\codecritic").resolve()

os.chdir(PROJECT_ROOT)  # Change current directory
sys.path.insert(0, str(PROJECT_ROOT))  # Ensure imports resolve correctly

print("Working directory is now:", Path.cwd())

from app.db.connection import DB_PATH, close_connection
from sqlalchemy import create_engine
from app.db.seeders.seed_scores import seed_score_providers
from app.db.seeders.seed_tools import seed_tool_providers
from app.db.seeders.seed_prompt_providers import seed_prompt_providers
from app.db.seeders.seed_prompts import seed_prompts
from app.db.seeders.seed_context_providers import seed_context_providers
from app.db.seeders.seed_agent_engine_providers import seed_agent_engine_providers
from app.db.base import Base
from sqlalchemy.orm import Session

# 🧹 Reset DB
close_connection()
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)

# 🌱 Seed database records
with Session(bind=engine) as session:
    seed_prompts(session)
    seed_prompt_providers(session)
    seed_tool_providers(session)
    seed_score_providers(session)
    seed_context_providers(session)
    seed_agent_engine_providers(session)

Working directory is now: C:\Repos\codecritic
Seeded AgentPrompt GUID: 9e0ae3bd-b87a-48e7-90ca-5732294c70f9
Seeded SystemPrompt GUID: 34490b0f-e944-4072-a109-df4c87d38ece
Seeded prompt provider configurations successfully.
Seeded tool configurations successfully.
Seeded score providers successfully.
Seeded context provider configurations successfully.
Seeded agent engine configurations successfully.


### 🔍 Step 2: Retrieve Agent Engine Provider ID from DB

In [3]:
from sqlalchemy import select
from app.db.models import AgentEngineProviderConfig

with Session(bind=engine) as session:
    record = session.execute(
        select(AgentEngineProviderConfig).where(AgentEngineProviderConfig.name == "Basic Agent Engine Provider")
    ).scalar_one()
    engine_provider_id = record.id
    print("✅ Found Agent Engine Provider ID:", engine_provider_id)

✅ Found Agent Engine Provider ID: 1


### 🏗️ Step 3: Instantiate Agent Engine from Factory

In [3]:
from app.factories.agent_engine_provider_factory import AgentEngineProviderFactory

engine_instance = AgentEngineProviderFactory.create(engine_provider_id)
print("✅ AgentEngineProvider instantiated:", engine_instance.__class__.__name__)

✅ AgentEngineProvider instantiated: BasicAgentEngineProvider


### 🧪 Step 4: Run Agent Engine with Test Prompt

In [4]:
response = engine_instance.run(prompt="What is the capital of France?", experiment_id="test_exp_001", round=1)
print("✅ AgentEngine Response:", response)

✅ AgentEngine Response: [Basic Response] You said: What is the capital of France?


### 📜 Step 5: Check Agent Engine Logs

In [6]:
from sqlalchemy import text

with engine.connect() as conn:
    rows = conn.execute(
        text("SELECT * FROM agent_engine_provider_log WHERE experiment_id='test_exp_001'")
    ).fetchall()

assert rows, "❌ No agent engine logs found."
print("✅ Logged Agent Engine Executions:")
for row in rows:
    print(row)

✅ Logged Agent Engine Executions:
(1, 'test_exp_001', 1, 'BasicAgentEngineProvider', 'mock-llm', 'What is the capital of France?', '[Basic Response] You said: What is the capital of France?', 1, None, 0, '2025-05-24T18:45:46.428220+00:00')
